# OCR-Engine-Agnostic Post-OCR Correction for Bengali Text Using Small Language Models

**Environment setup and pipeline dry run — Google Colab (CPU-only)**

This notebook prepares the experiment environment and verifies each pipeline stage on a small subset of data before the full evaluation sweep is run. It is intentionally CPU-only, matching the project's core constraint: the correction methods should work on free, commodity hardware, not require a GPU.

**Pipeline overview:**
1. Clone the project code from GitHub.
2. Install dependencies (OCR engines, Bengali language data, model libraries).
3. Load the dataset (British Library historical Bengali corpus) from Google Drive.
4. Run baseline OCR (Tesseract, EasyOCR) on the held-out development pages and verify the CER/WER scoring code against hand-written test cases.
5. Run a single small language model on one page as a sanity check of the zero-shot correction prompt, before committing to the full 5-model × 2-engine × 40-page evaluation.

Project repository: https://github.com/Ariffurrhmn/bengali-postocr-sllm

## 1. Clone the project repository

In [ ]:
import os

REPO_DIR = '/content/bengali-postocr-sllm'

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Ariffurrhmn/bengali-postocr-sllm.git {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists, pulling latest instead of re-cloning')
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 2. Install dependencies

Installs the Tesseract OCR engine (system package) and the pinned Python dependencies (`requirements.txt`): `pytesseract`, `easyocr`, `transformers`, `torch`, `jiwer`, and supporting libraries.

In [ ]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr
!pip install -q -r requirements.txt

### Bengali language data for Tesseract

The base Tesseract install does not include Bengali language data. The high-accuracy model (`tessdata_best`) is downloaded into a project-local `.tessdata/` folder, keeping the setup self-contained rather than depending on system-wide configuration.

In [ ]:
!mkdir -p .tessdata
!curl -sL -o .tessdata/ben.traineddata https://github.com/tesseract-ocr/tessdata_best/raw/main/ben.traineddata
!curl -sL -o .tessdata/eng.traineddata https://github.com/tesseract-ocr/tessdata_best/raw/main/eng.traineddata
!curl -sL -o .tessdata/osd.traineddata https://github.com/tesseract-ocr/tessdata_best/raw/main/osd.traineddata

## 3. Authenticate with Hugging Face

Two of the five correction models (Llama 3.2 1B, Gemma 2B) are gated and require an authenticated, license-accepted account to download.

**Before running this cell:** add your Hugging Face access token as a Colab secret (key icon in the left sidebar) named `HF_TOKEN`. This keeps the token out of the notebook's visible code and saved output.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

## 4. Load the dataset

The dataset (British Library historical Bengali corpus, 81 image + PAGE-XML page pairs) is not stored in the code repository. It is uploaded separately to Google Drive as `REID2019.zip` and unzipped here.

**Before running this cell:** upload `REID2019.zip` to a folder named `Dataset` in your Google Drive (`My Drive/Dataset/REID2019.zip`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATASET_ZIP = '/content/drive/My Drive/Dataset/REID2019.zip'
DATASET_DIR = '/content/Competition_dataset_ImagesPAGEXML'

!unzip -q -o "{DATASET_ZIP}" -d /content

import os
n_tif = len([f for f in os.listdir(DATASET_DIR) if f.lower().endswith('.tif')])
n_xml = len([f for f in os.listdir(DATASET_DIR) if f.lower().endswith('.xml')])
print(f'{n_tif} image files, {n_xml} PAGE-XML files found in {DATASET_DIR}')

## 5. Run baseline OCR on the development set

Runs both OCR engines (Tesseract, EasyOCR) over the 10 frozen development pages (`data/split_dev.txt`) and extracts each page's ground truth from its PAGE-XML file. Output is written to `results/ocr_dev.jsonl`.

In [ ]:
os.environ['TESSDATA_PREFIX'] = f'{REPO_DIR}/.tessdata'
os.environ['TESSERACT_CMD'] = 'tesseract'  # apt-get install puts tesseract on PATH

!cd {REPO_DIR}/ocr && python run_ocr.py --split dev --dataset-dir "{DATASET_DIR}"

## 6. Verify the evaluation code and score the baseline

First runs the hand-written CER/WER test cases (`eval/test_metrics.py`) to confirm the scoring code is correct before trusting it on real data, then scores the raw (uncorrected) OCR output from step 5 against ground truth.

In [ ]:
!cd {REPO_DIR}/eval && python test_metrics.py
print()
!cd {REPO_DIR}/eval && python score_baseline.py --split dev

## 7. Correction dry run

Runs a single model on a single page as a sanity check of the zero-shot correction prompt — catching obvious failure modes (wrong script, refusals, truncation) before committing to the full 5-model × 2-engine × 40-page evaluation sweep.

Starts with `titullm-1b` (Bengali-native, ungated) to avoid depending on gated-model access for this first check. Other valid values: `phi3-mini`, `llama3.2-1b`, `gemma-2b`, `banglat5`.

In [ ]:
!cd {REPO_DIR}/correction && python dry_run.py titullm-1b

## 8. Run OCR on a 15-page eval subsample

The full 40-page eval split, run across all models/approaches, is estimated at 60+ hours on free Colab CPU — infeasible. Instead, this project evaluates on a random 15-page subsample of the 40-page eval set (seed 403, reproducible), using the **chunked** approach (— the base paper's own segment-based method, ~250-token pieces rather than whole pages), across the 4 models that have run successfully so far (`titullm-1b`, `banglat5`, `llama3.2-1b`, `gemma-2b`; `phi3-mini` excluded — previously infeasible in reasonable time on free Colab CPU).

This cell runs OCR (Tesseract + EasyOCR) on those 15 pages, same as section 5 did for the dev set, but writing to **Google Drive** rather than the Colab VM's local disk — see the failsafe note in section 9 for why.

In [ ]:
RESULTS_DIR = '/content/drive/My Drive/bengali-postocr-results'
!mkdir -p "{RESULTS_DIR}"

!cd {REPO_DIR}/ocr && python run_ocr.py --split eval --dataset-dir "{DATASET_DIR}" --sample-n 15 --sample-seed 403 --out "{RESULTS_DIR}/ocr_eval.jsonl"

## 9. Full correction sweep on the eval subsample (long-running, resumable)

Runs the 4 working models across both OCR engines with the chunked approach, over the 15-page eval subsample from section 8.

**Resumable across disconnects, full runtime loss, or per-page errors:**
- Each individual result is written to the output file immediately after that page finishes (not batched at the end).
- Both the input OCR file (`--ocr-path`) and the output results file (`--out`) live on **Google Drive**, not the Colab VM's local disk — they survive a full runtime loss (idle disconnect reclaim, "delete runtime", or a power/connectivity loss on your end), not just a temporary reconnect to the same runtime.
- Re-running this exact cell — after reconnecting to the same runtime, or from a brand-new one via Runtime → Run all — reads what's already in the output file on Drive and skips every (page, engine, model, approach) combination already completed. It resumes, it does not restart or duplicate work.
- If a single page throws an error, it's logged in that page's record (`error` field) and the sweep continues to the next page rather than stopping the whole run.

**Scope:** 15 pages × 2 engines × 4 models × 1 approach (chunked) = 120 combinations. Estimated ~24 hours of compute based on dev-set per-page timing — expect this to span multiple sessions. Just re-run this cell each time you come back; it picks up where it left off.

In [ ]:
!cd {REPO_DIR}/correction && python run_sweep.py --split eval --models titullm-1b,banglat5,llama3.2-1b,gemma-2b --approaches chunked --engines tesseract,easyocr --ocr-path "{RESULTS_DIR}/ocr_eval.jsonl" --out "{RESULTS_DIR}/correction_eval.jsonl"

## 10. Score the eval-subsample results

Compares corrected output against the raw-OCR baseline, per (engine, model, approach) cell. Safe to re-run anytime while section 9's sweep is still in progress — it just scores whatever has been completed so far in the Drive-stored results file.

In [ ]:
!cd {REPO_DIR}/eval && python score_correction.py --split eval --ocr-path "{RESULTS_DIR}/ocr_eval.jsonl" --correction-path "{RESULTS_DIR}/correction_eval.jsonl"